In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 라이브러리

In [2]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# Torch 모델 구조
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT Model
# Fine Tuning
from transformers import AutoModel, AutoTokenizer
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput


# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [3]:
def extract_cls_embeddings(texts, tokenizer, model, batch_size=16):
    embeddings = []
    model.eval()
    device = model.device  # 모델이 할당된 디바이스 가져오기
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting [CLS] embeddings"):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
            inputs = {key: val.to(device) for key, val in inputs.items()}  # 모든 입력을 모델의 디바이스로 이동
            outputs = model(**inputs)
            cls_batch = outputs.last_hidden_state[:, 0, :]
            embeddings.append(cls_batch.cpu().numpy())  # GPU 메모리를 절약하기 위해 CPU로 이동 후 numpy 변환
    return np.vstack(embeddings)

In [4]:
def extract_first_image_urls(images_list):
    # 비어있거나 잘못된 값 방지
    if not images_list or not isinstance(images_list, list) or len(images_list) == 0:
        return pd.Series([None, None, None])
    img = images_list[0]
    return pd.Series([
        img.get('small_image_url'),
        img.get('medium_image_url'),
        img.get('large_image_url')
    ])

# 데이터셋 로딩

In [5]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_RQ_v2.pkl')
print(df.shape)
df.head()

(66718, 7)


,user_id,asin,text,rating,title_y,medium_image_url,ViT_emb_768
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,https://images-na.ssl-images-amazon.com/images...,"[-0.33046037, -0.5817304, -0.33555976, 0.27455..."
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,https://images-na.ssl-images-amazon.com/images...,"[-0.3084641, -0.29228264, -0.56403124, 0.76085..."
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,https://images-na.ssl-images-amazon.com/images...,"[-0.3357518, 0.674615, -0.31958464, 0.7886796,..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,https://images-na.ssl-images-amazon.com/images...,"[0.57578564, -0.32039246, -0.1263229, 0.048112..."
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,https://images-na.ssl-images-amazon.com/images...,"[0.0663303, -0.14241356, -0.5569954, -0.192086..."


# Load Model

## BERT

In [6]:
# load roberta
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
# roberta embedding 추출 후 concat
text_embeddings = extract_cls_embeddings(df['text'].tolist(), tokenizer, model, batch_size=16)
text_embeddings = pd.DataFrame({'bert_emb': [e for e in text_embeddings]})
df = pd.concat([df, text_embeddings], axis=1)

Extracting [CLS] embeddings: 100%|██████████| 4170/4170 [1:46:33<00:00,  1.53s/it]


In [ ]:
# 최종 Text Emb Dataset 저장
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_RQ_v3.pkl')